# 08 — Paper figure: pixel‑intensity histograms per exposure and HDR

Fig. 8 of the paper: normalised pixel‑intensity distributions of the nine
stacked exposures (with the inter‑polariser shifts applied) and of the two HDR
products, per polariser, with the number of frames stacked annotated above each
column.  Both HDR panels now come from this pipeline's own products.

Legacy source: `statistical_analysis_figures.ipynb` cell 11.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
from scipy import ndimage
table = pd.read_csv(config.FRAME_TABLE_CSV, dtype={"position": str})
good = table[table.use]          # frames actually stacked (radius filter + config.EXTRA_EXCLUDED_FRAMES)
counts = {pos: {e: int(((good.position == pos) & (good.inverse_exposure_time == e)).sum()) for e in config.INV_EXPOSURES}
          for pos in config.POLARIZER_POSITIONS}

def norm01(img):
    v = img[img > 0].ravel()
    return v / v.max()

hists = {pos: [] for pos in config.POLARIZER_POSITIONS}
for e in config.INV_EXPOSURES:
    d = fits.getdata(utils.stacked_filename(e))
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        hists[pos].append(norm01(ndimage.shift(np.asarray(d[i], np.float32), config.CHANNEL_SHIFTS[pos], order=1)))
for method in ("ldic", "expnorm"):
    for pos in config.POLARIZER_POSITIONS:
        hists[pos].append(norm01(fits.getdata(utils.hdr_filename(method, pos))))

In [ ]:
colors = {"1": "red", "2": "green", "3": "blue"}
labels = {p: f"Pol {p} ({config.POLARIZER_ANGLE_DEG[p]:+.0f}°)".replace("+0°", "0°") for p in config.POLARIZER_POSITIONS}
ncol = len(config.INV_EXPOSURES) + 2
fig, axes = plt.subplots(1, ncol, figsize=(24, 10), sharey=True)
for i, ax in enumerate(axes):
    for pos in config.POLARIZER_POSITIONS:
        ax.hist(hists[pos][i], bins=150, color=colors[pos], alpha=0.75, histtype="step", linewidth=2.0,
                label=labels[pos] if i == 0 else "", orientation="horizontal")
    if i < len(config.INV_EXPOSURES):
        e = config.INV_EXPOSURES[i]
        ax.set_title(f"1/{e} s", fontsize=17, fontweight="bold", pad=35)
        for x, pos in zip((0.15, 0.50, 0.85), config.POLARIZER_POSITIONS):
            ax.text(x, 1.02, f"{counts[pos][e]}", color=colors[pos], transform=ax.transAxes, ha="center", va="bottom", fontsize=16, fontweight="bold")
    else:
        ax.set_title(["LDIC HDR", "Exp. Norm HDR"][i - len(config.INV_EXPOSURES)], fontsize=17, fontweight="bold", pad=35)
    ax.set_xscale("log"); ax.set_ylim(0, 1.05)
    for s in ax.spines.values(): s.set_linewidth(1.5)
    ax.tick_params(axis="both", which="major", labelsize=15, width=1.5, length=6)
    ax.tick_params(axis="both", which="minor", width=1.0, length=3)
    if i == 0:
        ax.set_ylabel("Normalized Pixel Intensity", fontsize=18, fontweight="bold", labelpad=12)
        ax.legend(loc="upper right", fontsize=14, framealpha=0.95, edgecolor="black", fancybox=False)
    ax.grid(True, axis="y", alpha=0.4, linestyle="--", linewidth=1.0)
    ax.set_xlabel("Counts", fontsize=18, fontweight="bold")
plt.tight_layout(); plt.subplots_adjust(wspace=0.0)
fig.savefig(config.FIGURES_DIR / "histogram_normalized_exposures.png", bbox_inches="tight")
fig.savefig(config.FIGURES_DIR / "histogram_normalized_exposures.pdf", bbox_inches="tight")